In [56]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

spark = SparkSession.builder.appName("Week3_Order_Analysis").getOrCreate()
print("Spark Session created")

Spark Session created


In [57]:
customer_spark = spark.read.option("header", True).option("inferSchema", True).csv("customer.csv")

orders_spark = spark.read.option("header", True).option("inferSchema", True).csv("orders.csv")

delivery_spark = spark.read.option("header", True).option("inferSchema", True).csv("delivery.csv")

In [58]:
print("Customer Table")
customer_spark.show(truncate=False)

print("Orders Table")
orders_spark.show(truncate=False)

print("Delivery Table")
delivery_spark.show(truncate=False)

Customer Table
+-----------+-------------+----------+----------+
|customer_id|customer_name|city      |region    |
+-----------+-------------+----------+----------+
|C001       |Asha         |Salem     |Tamil Nadu|
|C002       |Ravi         |Chennai   |Tamil Nadu|
|C003       |Meena        |Bengaluru |Karnataka |
|C004       |Arun         |Coimbatore|Tamil Nadu|
|C005       |Sneha        |Mysuru    |Karnataka |
|C006       |Kiran        |Hyderabad |Telangana |
+-----------+-------------+----------+----------+

Orders Table
+--------+-----------+----------+------------+----------------+
|order_id|customer_id|order_date|order_amount|product_category|
+--------+-----------+----------+------------+----------------+
|O101    |C001       |2026-06-01|1200        |Electronics     |
|O102    |C002       |2026-06-02|850         |Books           |
|O103    |C003       |2026-06-03|2400        |Home            |
|O104    |C001       |2026-06-04|760         |Accessories     |
|O105    |C004       |2

In [59]:
print("Customer Schema")
customer_spark.printSchema()

print("Orders Schema")
orders_spark.printSchema()

print("Delivery Schema")
delivery_spark.printSchema()

Customer Schema
root
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- region: string (nullable = true)

Orders Schema
root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- order_amount: integer (nullable = true)
 |-- product_category: string (nullable = true)

Delivery Schema
root
 |-- delivery_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- delivery_date: date (nullable = true)
 |-- delivery_status: string (nullable = true)
 |-- delivery_issue: string (nullable = true)



In [60]:
customer_spark = customer_spark.dropDuplicates()

orders_spark = orders_spark.dropDuplicates()

delivery_spark = delivery_spark.dropDuplicates()

In [61]:
customer_spark = customer_spark.na.fill({
    "city": "Unknown",
    "region": "Unknown"})
customer_spark.show()

+-----------+-------------+----------+----------+
|customer_id|customer_name|      city|    region|
+-----------+-------------+----------+----------+
|       C005|        Sneha|    Mysuru| Karnataka|
|       C006|        Kiran| Hyderabad| Telangana|
|       C004|         Arun|Coimbatore|Tamil Nadu|
|       C002|         Ravi|   Chennai|Tamil Nadu|
|       C001|         Asha|     Salem|Tamil Nadu|
|       C003|        Meena| Bengaluru| Karnataka|
+-----------+-------------+----------+----------+



In [62]:
orders_spark = orders_spark.na.fill({
    "product_category": "Unknown",
    "order_amount": 0})
orders_spark.show()

+--------+-----------+----------+------------+----------------+
|order_id|customer_id|order_date|order_amount|product_category|
+--------+-----------+----------+------------+----------------+
|    O101|       C001|2026-06-01|        1200|     Electronics|
|    O104|       C001|2026-06-04|         760|     Accessories|
|    O108|       C002|2026-06-08|         400|           Books|
|    O106|       C005|2026-06-06|         950|          Beauty|
|    O102|       C002|2026-06-02|         850|           Books|
|    O105|       C004|2026-06-05|        1800|     Electronics|
|    O107|       C006|2026-06-07|        3200|       Furniture|
|    O103|       C003|2026-06-03|        2400|            Home|
+--------+-----------+----------+------------+----------------+



In [63]:
delivery_spark = delivery_spark.na.fill({
    "delivery_status": "Pending",
    "delivery_issue": "No issue reported"})
delivery_spark.show()

+-----------+--------+-------------+---------------+--------------------+
|delivery_id|order_id|delivery_date|delivery_status|      delivery_issue|
+-----------+--------+-------------+---------------+--------------------+
|       D201|    O101|   2026-06-05|      Delivered|       Late delivery|
|       D208|    O108|   2026-06-09|      Delivered|             On time|
|       D202|    O102|   2026-06-03|      Delivered|             On time|
|       D204|    O104|   2026-06-10|      Delivered|       Weather delay|
|       D205|    O105|   2026-06-06|      Delivered|             On time|
|       D206|    O106|   2026-06-12|      Delivered|       Late delivery|
|       D207|    O107|   2026-06-10|      Delivered|Delivery partner ...|
|       D203|    O103|         NULL|        Pending|       Address issue|
+-----------+--------+-------------+---------------+--------------------+



In [64]:
customer_orders = customer_spark.join(
    orders_spark, "customer_id", "inner")
customer_orders.show()

+-----------+-------------+----------+----------+--------+----------+------------+----------------+
|customer_id|customer_name|      city|    region|order_id|order_date|order_amount|product_category|
+-----------+-------------+----------+----------+--------+----------+------------+----------------+
|       C001|         Asha|     Salem|Tamil Nadu|    O101|2026-06-01|        1200|     Electronics|
|       C001|         Asha|     Salem|Tamil Nadu|    O104|2026-06-04|         760|     Accessories|
|       C002|         Ravi|   Chennai|Tamil Nadu|    O108|2026-06-08|         400|           Books|
|       C005|        Sneha|    Mysuru| Karnataka|    O106|2026-06-06|         950|          Beauty|
|       C002|         Ravi|   Chennai|Tamil Nadu|    O102|2026-06-02|         850|           Books|
|       C004|         Arun|Coimbatore|Tamil Nadu|    O105|2026-06-05|        1800|     Electronics|
|       C006|        Kiran| Hyderabad| Telangana|    O107|2026-06-07|        3200|       Furniture|


In [65]:
full_df = customer_orders.join(
    delivery_spark, "order_id", "left")
full_df.show()

+--------+-----------+-------------+----------+----------+----------+------------+----------------+-----------+-------------+---------------+--------------------+
|order_id|customer_id|customer_name|      city|    region|order_date|order_amount|product_category|delivery_id|delivery_date|delivery_status|      delivery_issue|
+--------+-----------+-------------+----------+----------+----------+------------+----------------+-----------+-------------+---------------+--------------------+
|    O101|       C001|         Asha|     Salem|Tamil Nadu|2026-06-01|        1200|     Electronics|       D201|   2026-06-05|      Delivered|       Late delivery|
|    O104|       C001|         Asha|     Salem|Tamil Nadu|2026-06-04|         760|     Accessories|       D204|   2026-06-10|      Delivered|       Weather delay|
|    O108|       C002|         Ravi|   Chennai|Tamil Nadu|2026-06-08|         400|           Books|       D208|   2026-06-09|      Delivered|             On time|
|    O106|       C005|

In [66]:
full_df = full_df.withColumn("order_date", to_timestamp(col("order_date"), "yyyy-MM-dd")) \
                 .withColumn("delivery_date", to_timestamp(col("delivery_date"), "yyyy-MM-dd"))
full_df.show()

+--------+-----------+-------------+----------+----------+-------------------+------------+----------------+-----------+-------------------+---------------+--------------------+
|order_id|customer_id|customer_name|      city|    region|         order_date|order_amount|product_category|delivery_id|      delivery_date|delivery_status|      delivery_issue|
+--------+-----------+-------------+----------+----------+-------------------+------------+----------------+-----------+-------------------+---------------+--------------------+
|    O101|       C001|         Asha|     Salem|Tamil Nadu|2026-06-01 00:00:00|        1200|     Electronics|       D201|2026-06-05 00:00:00|      Delivered|       Late delivery|
|    O104|       C001|         Asha|     Salem|Tamil Nadu|2026-06-04 00:00:00|         760|     Accessories|       D204|2026-06-10 00:00:00|      Delivered|       Weather delay|
|    O108|       C002|         Ravi|   Chennai|Tamil Nadu|2026-06-08 00:00:00|         400|           Books|  

In [67]:
full_df = full_df.withColumn(
    "delivery_date",
    when(col("delivery_date").isNull(), current_timestamp()
    ).otherwise(col("delivery_date"))
)

full_df.show()

+--------+-----------+-------------+----------+----------+-------------------+------------+----------------+-----------+--------------------+---------------+--------------------+
|order_id|customer_id|customer_name|      city|    region|         order_date|order_amount|product_category|delivery_id|       delivery_date|delivery_status|      delivery_issue|
+--------+-----------+-------------+----------+----------+-------------------+------------+----------------+-----------+--------------------+---------------+--------------------+
|    O101|       C001|         Asha|     Salem|Tamil Nadu|2026-06-01 00:00:00|        1200|     Electronics|       D201| 2026-06-05 00:00:00|      Delivered|       Late delivery|
|    O104|       C001|         Asha|     Salem|Tamil Nadu|2026-06-04 00:00:00|         760|     Accessories|       D204| 2026-06-10 00:00:00|      Delivered|       Weather delay|
|    O108|       C002|         Ravi|   Chennai|Tamil Nadu|2026-06-08 00:00:00|         400|           Boo

In [68]:
full_df = full_df.withColumn("delay_days", datediff(col("delivery_date"), col("order_date"))) \
                 .withColumn("delay_flag", when(col("delay_days") > 3, 1).otherwise(0))

full_df.show()

+--------+-----------+-------------+----------+----------+-------------------+------------+----------------+-----------+--------------------+---------------+--------------------+----------+----------+
|order_id|customer_id|customer_name|      city|    region|         order_date|order_amount|product_category|delivery_id|       delivery_date|delivery_status|      delivery_issue|delay_days|delay_flag|
+--------+-----------+-------------+----------+----------+-------------------+------------+----------------+-----------+--------------------+---------------+--------------------+----------+----------+
|    O101|       C001|         Asha|     Salem|Tamil Nadu|2026-06-01 00:00:00|        1200|     Electronics|       D201| 2026-06-05 00:00:00|      Delivered|       Late delivery|         4|         1|
|    O104|       C001|         Asha|     Salem|Tamil Nadu|2026-06-04 00:00:00|         760|     Accessories|       D204| 2026-06-10 00:00:00|      Delivered|       Weather delay|         6|       

In [69]:
delayed_orders = full_df.filter(col("delay_flag") == 1)
delayed_orders.show()

+--------+-----------+-------------+---------+----------+-------------------+------------+----------------+-----------+--------------------+---------------+--------------+----------+----------+
|order_id|customer_id|customer_name|     city|    region|         order_date|order_amount|product_category|delivery_id|       delivery_date|delivery_status|delivery_issue|delay_days|delay_flag|
+--------+-----------+-------------+---------+----------+-------------------+------------+----------------+-----------+--------------------+---------------+--------------+----------+----------+
|    O101|       C001|         Asha|    Salem|Tamil Nadu|2026-06-01 00:00:00|        1200|     Electronics|       D201| 2026-06-05 00:00:00|      Delivered| Late delivery|         4|         1|
|    O104|       C001|         Asha|    Salem|Tamil Nadu|2026-06-04 00:00:00|         760|     Accessories|       D204| 2026-06-10 00:00:00|      Delivered| Weather delay|         6|         1|
|    O106|       C005|        

In [70]:
region_delay = full_df.groupBy("region").agg(
    count("order_id").alias("total_orders"),
    count(when(col("delay_flag") == 1, 1)).alias("delayed_orders")
).orderBy(desc("delayed_orders"))

region_delay.show()

+----------+------------+--------------+
|    region|total_orders|delayed_orders|
+----------+------------+--------------+
| Karnataka|           2|             2|
|Tamil Nadu|           5|             2|
| Telangana|           1|             0|
+----------+------------+--------------+



In [71]:
region_sales = full_df.groupBy("region").agg(
    sum("order_amount").alias("total_sales"),
    avg("order_amount").alias("avg_order_value"),
    count("order_id").alias("order_count")
).orderBy(desc("total_sales"))

region_sales.show()

+----------+-----------+---------------+-----------+
|    region|total_sales|avg_order_value|order_count|
+----------+-----------+---------------+-----------+
|Tamil Nadu|       5010|         1002.0|          5|
| Karnataka|       3350|         1675.0|          2|
| Telangana|       3200|         3200.0|          1|
+----------+-----------+---------------+-----------+



In [72]:
delivery_summary = full_df.groupBy("delivery_issue").count().orderBy(desc("count"))
delivery_summary.show()

+--------------------+-----+
|      delivery_issue|count|
+--------------------+-----+
|             On time|    3|
|       Late delivery|    2|
|       Weather delay|    1|
|       Address issue|    1|
|Delivery partner ...|    1|
+--------------------+-----+



In [73]:
customer_delay_summary = full_df.groupBy("customer_id", "customer_name").agg(
    count("order_id").alias("total_orders"),
    sum("delay_flag").alias("delayed_orders"),
    sum("order_amount").alias("total_spend")
).orderBy(desc("delayed_orders"))

customer_delay_summary.show()

+-----------+-------------+------------+--------------+-----------+
|customer_id|customer_name|total_orders|delayed_orders|total_spend|
+-----------+-------------+------------+--------------+-----------+
|       C001|         Asha|           2|             2|       1960|
|       C005|        Sneha|           1|             1|        950|
|       C003|        Meena|           1|             1|       2400|
|       C002|         Ravi|           2|             0|       1250|
|       C006|        Kiran|           1|             0|       3200|
|       C004|         Arun|           1|             0|       1800|
+-----------+-------------+------------+--------------+-----------+



In [74]:
print("Total Records:", full_df.count())
print("\nTotal Delayed Orders:", delayed_orders.count())
print("\nAverage Order Value:", full_df.select(avg("order_amount")).first()[0])

Total Records: 8

Total Delayed Orders: 4

Average Order Value: 1445.0


In [75]:
full_df.select("order_id","customer_name","region","order_amount","delay_days","delay_flag").show()

+--------+-------------+----------+------------+----------+----------+
|order_id|customer_name|    region|order_amount|delay_days|delay_flag|
+--------+-------------+----------+------------+----------+----------+
|    O101|         Asha|Tamil Nadu|        1200|         4|         1|
|    O104|         Asha|Tamil Nadu|         760|         6|         1|
|    O108|         Ravi|Tamil Nadu|         400|         1|         0|
|    O106|        Sneha| Karnataka|         950|         6|         1|
|    O102|         Ravi|Tamil Nadu|         850|         1|         0|
|    O105|         Arun|Tamil Nadu|        1800|         1|         0|
|    O107|        Kiran| Telangana|        3200|         3|         0|
|    O103|        Meena| Karnataka|        2400|        24|         1|
+--------+-------------+----------+------------+----------+----------+



In [76]:
full_df.write.mode("overwrite").option("header", True).csv("week3_order_analysis_output_csv")

full_df.write.mode("overwrite").parquet("week3_order_analysis_output_parquet")

print("Output saved as CSV and Parquet")

Output saved as CSV and Parquet
